## BASE

In [1]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
import time
from tqdm.notebook import tqdm
from collections import defaultdict

In [ ]:
# DATA_ROOT = "/home/alex/internship/datasets/aqua20/data/aqua20"
# DATA_ROOT = "/Users/alex/Developpement/Internship/datasets/aqua20/data/aqua20"
DATA_ROOT = "/lustre/fswork/projects/rech/rbw/ucw75ke/datasets/aqua20/data/aqua20"
NUM_CLASSES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252

In [3]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [4]:
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False
backbone = backbone.to(DEVICE)

Using cache found in /Users/alex/.cache/torch/hub/facebookresearch_dinov2_main
/Users/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [25]:
def extract_features(loader, desc="Extracting features"):
    all_feats, all_labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc, leave=False):
            all_feats.append(backbone(x.to(DEVICE)).cpu())
            all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)

def train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                       epochs=50, lr=1e-3, eval_every=10):
    head = nn.Linear(768, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_feat_loader = DataLoader(TensorDataset(train_feats, train_labels), batch_size=256, shuffle=True)
    test_feat_loader = DataLoader(TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False)

    history: dict[int, dict[str, float]] = {}

    for epoch in tqdm(range(epochs), desc="Training"):
        head.train()
        total_loss, correct, total = 0.0, 0, 0

        for feats, y in train_feat_loader:
            feats, y = feats.to(DEVICE), y.to(DEVICE)
            logits = head(feats)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(y)

        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            head.eval()
            all_preds, all_labels_val = [], []
            with torch.no_grad():
                for feats, y in test_feat_loader:
                    all_preds.append(head(feats.to(DEVICE)).argmax(dim=1).cpu())
                    all_labels_val.append(y)
            all_preds = torch.cat(all_preds).numpy()
            all_labels_val = torch.cat(all_labels_val).numpy()

            history[epoch + 1] = {
                "loss": total_loss / total,
                "train_acc": correct / total,
                "f1_macro": f1_score(all_labels_val, all_preds, average="macro"),
                "f1_weighted": f1_score(all_labels_val, all_preds, average="weighted"),
            }

            m = history[epoch + 1]
            tqdm.write(
                f"Epoch {epoch+1:>3}/{epochs} | Loss: {m['loss']:.4f} | "
                f"Train Acc: {m['train_acc']*100:.1f}% | "
                f"F1 Macro: {m['f1_macro']*100:.1f}% | F1 Weighted: {m['f1_weighted']*100:.1f}%"
            )
    return head, history

In [9]:
test_ds    = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=transform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
test_feats,    test_labels    = extract_features(test_loader,    "Test")

Test:   0%|          | 0/26 [00:00<?, ?it/s]

## Baselines

### Full Data

In [10]:
full_train = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=transform)
full_loader = DataLoader(full_train, batch_size=64, shuffle=True)

In [13]:
print("\nTraining on full data...")
start_time = time.time()
full_feats,    full_labels    = extract_features(full_loader,    "Full train")
head_full, history_full = train_linear_probe(full_feats, full_labels, test_feats, test_labels,
                                epochs=50, eval_every=10)
full_total_time = time.time() - start_time
print(f"Training time: {full_total_time:.6f} seconds")


Training on full data...


Full train:   0%|          | 0/103 [00:00<?, ?it/s]

Training:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch  10/50 | Loss: 0.1458 | Train Acc: 95.0% | F1 Macro: 89.6% | F1 Weighted: 90.9%


IndexError: list index out of range

## Setup multiple


In [ ]:
DISTILLED_BASE_DIR = "../logged_files/distillation/aqua20/dinov2_vitb"

In [ ]:
class DistilledRun():

    def __init__(self, distilled_run: str, batch_size: int = 20):
        self.distilled_run: str = distilled_run
        self.syn_data: dict[str, torch.Tensor] = torch.load(
            f"{DISTILLED_BASE_DIR}/{distilled_run}/data.pth", 
            map_location="cpu"
        )
        self.distill_loader: DataLoader = DataLoader(
            TensorDataset(self.syn_data["images"], self.syn_data["labels"]),
            batch_size=batch_size, shuffle=False
        )
        self.total_time: float = 0.0
        self.head: nn.Linear | None = None
        self.history: dict | None = None

    def run_training(self, test_feats, test_labels, epochs=50, lr=1e-3, eval_every=10):
        start_time = time.time()
        feats, labels = extract_features(self.distill_loader, desc=f"Extracting features for {self.distilled_run}")
        self.head, self.history = train_linear_probe(feats, labels, test_feats, test_labels,
                                  epochs=epochs, lr=lr, eval_every=eval_every)
        self.total_time = time.time() - start_time
    
    def to_dict(self):
        return {"total_time": self.total_time, "history": self.history}
    
    def __str__(self):
        return f"DistilledRun(run={self.distilled_run}, total_time={self.total_time:.2f}s)"

In [ ]:
import re
from pathlib import Path

distilled_run_regex = re.compile(r"^distill_aqua20_h100_\d+ipc(?:_(?:physics|seathru))?$")

distilled_runs: list[DistilledRun] = [
    DistilledRun(p.name)
    for p in sorted(Path(DISTILLED_BASE_DIR).iterdir())
    if p.is_dir() and distilled_run_regex.match(p.name)
]

print(f"Found {len(distilled_runs)} distilled runs:")

In [ ]:
for run in distilled_runs:
    print(f"Running training for {run.distilled_run}...")
    run.run_training(epochs=50, lr=1e-3, eval_every=10)
    print(f"Finished {run.distilled_run} in {run.total_time:.2f}s\n")

In [ ]:
global_history = {run.distilled_run: run.to_dict() for run in distilled_runs}
global_history["full_data"] = {"total_time": full_total_time, "history": history_full}

In [ ]:
print(global_history)

In [29]:
data = {'distill_aqua20_h100_10ipc': {'total_time': 2.965365409851074, 'history': {10: {'loss': 0.3259713053703308, 'train_acc': 0.945, 'f1_macro': 0.7521001358035042, 'f1_weighted': 0.790583989728998}, 20: {'loss': 0.05851832404732704, 'train_acc': 1.0, 'f1_macro': 0.8006397578096701, 'f1_weighted': 0.833651294928404}, 30: {'loss': 0.01834903471171856, 'train_acc': 1.0, 'f1_macro': 0.8029078023923575, 'f1_weighted': 0.8349889045940574}, 40: {'loss': 0.009410875849425793, 'train_acc': 1.0, 'f1_macro': 0.8010909459854403, 'f1_weighted': 0.8335733999954215}, 50: {'loss': 0.006401942577213049, 'train_acc': 1.0, 'f1_macro': 0.8026606368210544, 'f1_weighted': 0.8338014572333198}}}, 'distill_aqua20_h100_10ipc_physics': {'total_time': 1.400111198425293, 'history': {10: {'loss': 0.16167953610420227, 'train_acc': 0.985, 'f1_macro': 0.7844243539331346, 'f1_weighted': 0.8315767239713026}, 20: {'loss': 0.01865849830210209, 'train_acc': 1.0, 'f1_macro': 0.8232700705096778, 'f1_weighted': 0.8530885409614616}, 30: {'loss': 0.005976757500320673, 'train_acc': 1.0, 'f1_macro': 0.8256081643375115, 'f1_weighted': 0.8527841405336536}, 40: {'loss': 0.0034733996726572514, 'train_acc': 1.0, 'f1_macro': 0.8256585364652584, 'f1_weighted': 0.8534432030352289}, 50: {'loss': 0.0025402591563761234, 'train_acc': 1.0, 'f1_macro': 0.8270738354062981, 'f1_weighted': 0.8547673190846685}}}, 'distill_aqua20_h100_10ipc_seathru': {'total_time': 1.460906982421875, 'history': {10: {'loss': 0.15609322488307953, 'train_acc': 0.995, 'f1_macro': 0.7788358105847859, 'f1_weighted': 0.8072496377507808}, 20: {'loss': 0.021876506507396698, 'train_acc': 1.0, 'f1_macro': 0.8139824799490979, 'f1_weighted': 0.832402871664572}, 30: {'loss': 0.007455546408891678, 'train_acc': 1.0, 'f1_macro': 0.8185313466036671, 'f1_weighted': 0.8334407758840905}, 40: {'loss': 0.0041726697236299515, 'train_acc': 1.0, 'f1_macro': 0.820445915660571, 'f1_weighted': 0.8345391592188947}, 50: {'loss': 0.0030187570955604315, 'train_acc': 1.0, 'f1_macro': 0.8226603414812406, 'f1_weighted': 0.8349897667225181}}}, 'distill_aqua20_h100_1ipc': {'total_time': 0.28571200370788574, 'history': {10: {'loss': 0.02784910425543785, 'train_acc': 1.0, 'f1_macro': 0.7163298756254685, 'f1_weighted': 0.7613785838899556}, 20: {'loss': 0.002004249719902873, 'train_acc': 1.0, 'f1_macro': 0.7567293378449789, 'f1_weighted': 0.7719398595963664}, 30: {'loss': 0.0006399654666893184, 'train_acc': 1.0, 'f1_macro': 0.7554439929261975, 'f1_weighted': 0.7689369868849327}, 40: {'loss': 0.00039559538708999753, 'train_acc': 1.0, 'f1_macro': 0.754058628664474, 'f1_weighted': 0.7694031115300511}, 50: {'loss': 0.00032019815989769995, 'train_acc': 1.0, 'f1_macro': 0.7565567841937468, 'f1_weighted': 0.7704413328657629}}}, 'distill_aqua20_h100_1ipc_physics': {'total_time': 0.28588151931762695, 'history': {10: {'loss': 0.026921581476926804, 'train_acc': 1.0, 'f1_macro': 0.7098784074762643, 'f1_weighted': 0.7137501421730216}, 20: {'loss': 0.0014805661048740149, 'train_acc': 1.0, 'f1_macro': 0.7458897316518674, 'f1_weighted': 0.7364933100789468}, 30: {'loss': 0.0005236336146481335, 'train_acc': 1.0, 'f1_macro': 0.7548138204439473, 'f1_weighted': 0.7428664087464195}, 40: {'loss': 0.0003384860174264759, 'train_acc': 1.0, 'f1_macro': 0.7562490750387264, 'f1_weighted': 0.7452585374388847}, 50: {'loss': 0.00027248269179835916, 'train_acc': 1.0, 'f1_macro': 0.7552858686039802, 'f1_weighted': 0.747166962676912}}}, 'distill_aqua20_h100_1ipc_seathru': {'total_time': 0.2844233512878418, 'history': {10: {'loss': 0.033954374492168427, 'train_acc': 1.0, 'f1_macro': 0.6805965249501099, 'f1_weighted': 0.7251909743384203}, 20: {'loss': 0.0021454752422869205, 'train_acc': 1.0, 'f1_macro': 0.7304469622450822, 'f1_weighted': 0.7584865926624693}, 30: {'loss': 0.0007053919835016131, 'train_acc': 1.0, 'f1_macro': 0.7367712888712052, 'f1_weighted': 0.7653287194376716}, 40: {'loss': 0.0004234728985466063, 'train_acc': 1.0, 'f1_macro': 0.7390821283019754, 'f1_weighted': 0.769158900365058}, 50: {'loss': 0.0003338521346449852, 'train_acc': 1.0, 'f1_macro': 0.7386120965740907, 'f1_weighted': 0.7680672456768867}}}, 'distill_aqua20_h100_3ipc': {'total_time': 0.5381758213043213, 'history': {10: {'loss': 0.08780981600284576, 'train_acc': 1.0, 'f1_macro': 0.7040545586263023, 'f1_weighted': 0.771496688735948}, 20: {'loss': 0.006556236185133457, 'train_acc': 1.0, 'f1_macro': 0.7493808193881863, 'f1_weighted': 0.8011682607439404}, 30: {'loss': 0.0022019455209374428, 'train_acc': 1.0, 'f1_macro': 0.7585240618588507, 'f1_weighted': 0.8091272524414969}, 40: {'loss': 0.001331381849013269, 'train_acc': 1.0, 'f1_macro': 0.7614619016364007, 'f1_weighted': 0.8121214726457798}, 50: {'loss': 0.0010368229122832417, 'train_acc': 1.0, 'f1_macro': 0.7653435511911317, 'f1_weighted': 0.8143246161615869}}}, 'distill_aqua20_h100_3ipc_physics': {'total_time': 0.5316946506500244, 'history': {10: {'loss': 0.07913605123758316, 'train_acc': 1.0, 'f1_macro': 0.7274444516172035, 'f1_weighted': 0.7774133142494543}, 20: {'loss': 0.005165517795830965, 'train_acc': 1.0, 'f1_macro': 0.7601399160728988, 'f1_weighted': 0.8086294566375688}, 30: {'loss': 0.001496823038905859, 'train_acc': 1.0, 'f1_macro': 0.7698360179767667, 'f1_weighted': 0.8149800813727426}, 40: {'loss': 0.0008761499775573611, 'train_acc': 1.0, 'f1_macro': 0.7775963110102501, 'f1_weighted': 0.8171363667378811}, 50: {'loss': 0.0006788679165765643, 'train_acc': 1.0, 'f1_macro': 0.776056483112734, 'f1_weighted': 0.8170649909241805}}}, 'distill_aqua20_h100_3ipc_seathru': {'total_time': 0.5287163257598877, 'history': {10: {'loss': 0.06490834057331085, 'train_acc': 1.0, 'f1_macro': 0.759817829026889, 'f1_weighted': 0.7831042928677286}, 20: {'loss': 0.004992331378161907, 'train_acc': 1.0, 'f1_macro': 0.8083996201506916, 'f1_weighted': 0.8168324022149684}, 30: {'loss': 0.0017670977395027876, 'train_acc': 1.0, 'f1_macro': 0.8194559720933177, 'f1_weighted': 0.8237909277563057}, 40: {'loss': 0.0010054364101961255, 'train_acc': 1.0, 'f1_macro': 0.8216287342213902, 'f1_weighted': 0.8335372280555078}, 50: {'loss': 0.0007521684165112674, 'train_acc': 1.0, 'f1_macro': 0.8223914007811611, 'f1_weighted': 0.8354132270941544}}}, 'distill_aqua20_h100_5ipc': {'total_time': 0.7744643688201904, 'history': {10: {'loss': 0.13420911133289337, 'train_acc': 1.0, 'f1_macro': 0.7617883821721627, 'f1_weighted': 0.8033497121045937}, 20: {'loss': 0.013496313244104385, 'train_acc': 1.0, 'f1_macro': 0.7965479167901076, 'f1_weighted': 0.8401902279991242}, 30: {'loss': 0.0047282506711781025, 'train_acc': 1.0, 'f1_macro': 0.8096747641728215, 'f1_weighted': 0.8458842164355533}, 40: {'loss': 0.0027760935481637716, 'train_acc': 1.0, 'f1_macro': 0.8141597703124145, 'f1_weighted': 0.8469594269308482}, 50: {'loss': 0.002090690890327096, 'train_acc': 1.0, 'f1_macro': 0.816098811916721, 'f1_weighted': 0.8483705305377409}}}, 'distill_aqua20_h100_5ipc_physics': {'total_time': 0.7781901359558105, 'history': {10: {'loss': 0.07161840796470642, 'train_acc': 1.0, 'f1_macro': 0.7540812723402736, 'f1_weighted': 0.7458468596183399}, 20: {'loss': 0.005489781964570284, 'train_acc': 1.0, 'f1_macro': 0.8022751738233497, 'f1_weighted': 0.8256814823535148}, 30: {'loss': 0.001864151214249432, 'train_acc': 1.0, 'f1_macro': 0.8030362074054679, 'f1_weighted': 0.8264686270918279}, 40: {'loss': 0.0011323407525196671, 'train_acc': 1.0, 'f1_macro': 0.8052307217946766, 'f1_weighted': 0.8257059138633484}, 50: {'loss': 0.0008863084949553013, 'train_acc': 1.0, 'f1_macro': 0.8061093244238592, 'f1_weighted': 0.8246509262831085}}}, 'distill_aqua20_h100_5ipc_seathru': {'total_time': 0.7738118171691895, 'history': {10: {'loss': 0.10614412277936935, 'train_acc': 0.99, 'f1_macro': 0.735532081663362, 'f1_weighted': 0.8035768910605141}, 20: {'loss': 0.00850427895784378, 'train_acc': 1.0, 'f1_macro': 0.7970121440523478, 'f1_weighted': 0.8293918786378786}, 30: {'loss': 0.0027739594224840403, 'train_acc': 1.0, 'f1_macro': 0.8147217198780397, 'f1_weighted': 0.8329055578195171}, 40: {'loss': 0.0016624503768980503, 'train_acc': 1.0, 'f1_macro': 0.8179521771416871, 'f1_weighted': 0.8386023795236909}, 50: {'loss': 0.0012899445137009025, 'train_acc': 1.0, 'f1_macro': 0.821708673168508, 'f1_weighted': 0.8400342876180269}}}, 'full_data': {'total_time': 239.66800236701965, 'history': {10: {'loss': 0.14508167737766925, 'train_acc': 0.9512120750114347, 'f1_macro': 0.9077540870896303, 'f1_weighted': 0.9067963593125947}, 20: {'loss': 0.10068756274573414, 'train_acc': 0.967982924226254, 'f1_macro': 0.9051105451015727, 'f1_weighted': 0.9122326328971282}, 30: {'loss': 0.07851380537692722, 'train_acc': 0.9753011129745388, 'f1_macro': 0.9029790263915259, 'f1_weighted': 0.90911913525344}, 40: {'loss': 0.0642821592203381, 'train_acc': 0.9818569903948773, 'f1_macro': 0.900598102913075, 'f1_weighted': 0.9060115565571948}, 50: {'loss': 0.05435644316566956, 'train_acc': 0.9853636225034303, 'f1_macro': 0.8958223772760114, 'f1_weighted': 0.9043401076013157}}}}


In [30]:
import re
import pandas as pd

def parse_run_name(name: str) -> tuple[int | None, str]:
    """Extrait (ipc, variante) du nom de run. full_data → (None, 'full')."""
    if name == "full_data":
        return None, "full"
    m = re.match(r"distill_aqua20_h100_(\d+)ipc(?:_(physics|seathru))?$", name)
    ipc = int(m.group(1))
    variant = m.group(2) or "baseline"
    return ipc, variant

rows = []
for run_name, run_data in data.items():
    ipc, variant = parse_run_name(run_name)
    last_epoch = max(run_data["history"].keys())
    metrics = run_data["history"][last_epoch]
    rows.append({
        "IPC": ipc,
        "Variant": variant,
        "F1 macro (%)": metrics["f1_macro"] * 100,
        "F1 weighted (%)": metrics["f1_weighted"] * 100,
        "Train acc (%)": metrics["train_acc"] * 100,
        "Time (s)": run_data["total_time"],
    })

df = pd.DataFrame(rows)

variant_order = ["baseline", "physics", "seathru", "full"]
df["Variant"] = pd.Categorical(df["Variant"], categories=variant_order, ordered=True)
df = df.sort_values(["IPC", "Variant"], na_position="last").reset_index(drop=True)

print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

  IPC  Variant  F1 macro (%)  F1 weighted (%)  Train acc (%)  Time (s)
 1.00 baseline         75.66            77.04         100.00      0.29
 1.00  physics         75.53            74.72         100.00      0.29
 1.00  seathru         73.86            76.81         100.00      0.28
 3.00 baseline         76.53            81.43         100.00      0.54
 3.00  physics         77.61            81.71         100.00      0.53
 3.00  seathru         82.24            83.54         100.00      0.53
 5.00 baseline         81.61            84.84         100.00      0.77
 5.00  physics         80.61            82.47         100.00      0.78
 5.00  seathru         82.17            84.00         100.00      0.77
10.00 baseline         80.27            83.38         100.00      2.97
10.00  physics         82.71            85.48         100.00      1.40
10.00  seathru         82.27            83.50         100.00      1.46
  NaN     full         89.58            90.43          98.54    239.67
